# Análisis Semántico con BETO (BERT para español)

## Imports y configuración

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.spatial.distance import cosine
import torch
from transformers import BertTokenizer, BertModel, pipeline
import sys
sys.path.append(os.path.abspath('../../../../../..'))
from src.data.mongo_storage import _get_default_collection, guardar_embeddings

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FIGS   = Path('../../..') / 'data' / 'figures'
MODELS = Path('../../..') / 'data' / 'models'
for d in [FIGS, MODELS]: d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.facecolor':'white','axes.grid':True,'grid.alpha':0.3,'font.size':11})
PALETTE = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860','#DA8BC3','#8C8C8C','#CCB974','#64B5CD']
print(f'✓ Imports OK — dispositivo: {DEVICE}')


## Cargar corpus desde MongoDB

In [ ]:
col = _get_default_collection()

# Cargar todas las canciones con sus Lyrics
docs = list(col.find(
    {'Lyrics': {'$ne': None}},
    {'_id': 1, 'Song': 1, 'Artist': 1, 'Genre': 1,
     'Song year': 1, 'Lyrics': 1, 'Language': 1}
))
df = pd.DataFrame(docs)
df = df.dropna(subset=['Lyrics', 'Genre']).copy()
df['Lyrics'] = df['Lyrics'].astype(str)
df = df[df['Lyrics'].str.len() > 50].reset_index(drop=True)

generos_validos = df['Genre'].value_counts()
generos_validos = generos_validos[generos_validos >= 20].index.tolist()
df = df[df['Genre'].isin(generos_validos)].reset_index(drop=True)

print(f'✓ {len(df):,} canciones cargadas desde MongoDB | {df["Genre"].nunique()} géneros')


## Muestra estratificada para BETO

BETO es costoso computacionalmente; se usa una muestra balanceada por género.

In [ ]:
# Ajusta MAX_POR_GENERO según tu hardware:
#   CPU lenta  → 50 | CPU normal → 100 | GPU → 200+
MAX_POR_GENERO = 100

df_sample = (
    df.groupby('Genre', group_keys=False)
      .apply(lambda x: x.sample(min(len(x), MAX_POR_GENERO), random_state=42))
      .reset_index(drop=True)
)
LYRICS = df_sample['Lyrics'].tolist()
GENRES = df_sample['Genre'].tolist()
print(f'✓ Muestra para BETO: {len(df_sample):,} canciones')
print(df_sample['Genre'].value_counts().to_string())


## Cargar modelo BETO

In [ ]:
# dccuchile/bert-base-spanish-wwm-cased → BETO (BERT entrenado en español)
# Para corpus en inglés cambia a: 'bert-base-uncased'
MODEL_NAME = 'dccuchile/bert-base-spanish-wwm-cased'

print(f'Cargando {MODEL_NAME}...')
print('(Primera vez: descarga ~420 MB)\n')

tokenizer_beto = BertTokenizer.from_pretrained(MODEL_NAME)
model_beto = BertModel.from_pretrained(MODEL_NAME).to(DEVICE)
model_beto.eval()

print(f'✓ BETO cargado en {DEVICE}')
print(f'   Parámetros  : {sum(p.numel() for p in model_beto.parameters()):,}')
print(f'   Dim. output : {model_beto.config.hidden_size}')


## Funciones de embedding

In [ ]:
def beto_embedding(text: str) -> np.ndarray:
    """Embedding BETO — vector [CLS] del último hidden state."""
    inputs = tokenizer_beto(
        text[:1000], return_tensors='pt',
        truncation=True, max_length=512, padding='max_length',
    ).to(DEVICE)
    with torch.no_grad():
        out = model_beto(**inputs)
    return out.last_hidden_state[:, 0, :].squeeze().cpu().numpy()


def beto_embeddings_batch(texts: list, batch_size: int = 16) -> np.ndarray:
    result = []
    n = len(texts)
    for i in range(0, n, batch_size):
        batch = texts[i:i + batch_size]
        result.extend([beto_embedding(t) for t in batch])
        done = min(i + batch_size, n)
        bar  = '█' * int(done/n*25) + '░' * (25 - int(done/n*25))
        print(f'  [{bar}] {done}/{n}', end='\r')
    print()
    return np.array(result)


def word_contextual_embedding(sentence: str, word: str) -> np.ndarray:
    """Embedding contextual de una palabra dentro de su oración."""
    inputs = tokenizer_beto(
        sentence, return_tensors='pt', truncation=True, max_length=128
    ).to(DEVICE)
    tokens = tokenizer_beto.convert_ids_to_tokens(inputs['input_ids'][0])
    idxs = [i for i, tok in enumerate(tokens)
            if word.lower() in tok.lower().replace('##', '')]
    with torch.no_grad():
        out = model_beto(**inputs)
    hidden = out.last_hidden_state[0].cpu().numpy()
    return hidden[idxs].mean(axis=0) if idxs else hidden.mean(axis=0)


# Prueba rápida
e = beto_embedding('hola mundo esta es una canción de prueba')
print(f'✓ Función lista — embedding shape: {e.shape}')


## Generar embeddings BETO y guardar en MongoDB

In [ ]:
BETO_FILE   = MODELS / 'beto_embeddings.npy'
SAMPLE_FILE = MODELS / 'beto_corpus_sample.csv'

# Caché en disco para no recalcular
if BETO_FILE.exists():
    beto_embs = np.load(BETO_FILE)
    if len(beto_embs) == len(df_sample):
        print(f'✓ Embeddings cargados desde caché: {beto_embs.shape}')
    else:
        BETO_FILE.unlink()
        print('Tamaño diferente, recalculando...')

if not BETO_FILE.exists():
    print(f'Generando embeddings BETO para {len(LYRICS)} canciones...')
    print(f'(Estimado en CPU: ~{len(LYRICS)//60} min)')
    beto_embs = beto_embeddings_batch(LYRICS, batch_size=8)
    np.save(BETO_FILE, beto_embs)
    df_sample.to_csv(SAMPLE_FILE, index=False)
    print(f'✓ Embeddings guardados en disco: {beto_embs.shape}')

# Guardar en MongoDB (actualiza campo beto_cls sin pisar word2vec_avg)
actualizados = 0
for i, (_, row) in enumerate(df_sample.iterrows()):
    # Recuperar word2vec_avg ya guardado para no pisarlo
    doc = col.find_one({'_id': row['_id']}, {'embeddings': 1})
    w2v_avg = []
    if doc and doc.get('embeddings') and doc['embeddings'].get('word2vec_avg'):
        w2v_avg = doc['embeddings']['word2vec_avg']

    ok = guardar_embeddings(
        song_id=row['_id'],
        word2vec_avg=w2v_avg,
        beto_cls=beto_embs[i].tolist(),
    )
    if ok: actualizados += 1

print(f'✓ beto_cls guardado en MongoDB: {actualizados}/{len(df_sample)} canciones')
print(f'Shape final: {beto_embs.shape}')


## Análisis de polisemia contextual

In [ ]:
POLISEMIA = {
    'fuego': [
        ('Rock',       'la guitarra está en llamas esta noche quemamos el escenario con el sonido'),
        ('Pop',        'tu amor prendió fuego a mi corazón cada vez que sonríes'),
        ('Hip-Hop',    'respondemos con fuego sin retroceder las calles nos tienen listos para la guerra'),
        ('Metal',      'fuego y azufre llueven del infierno consumiendo toda la luz'),
        ('Electronic', 'enciende el sintetizador el bajo cae fuerte en el club esta noche'),
    ],
    'corazón': [
        ('Pop',        'mi corazón late más rápido cuando me abrazas bajo las estrellas'),
        ('Rock',       'corazón de oro alma de hierro luchamos hasta el final amargo'),
        ('Hip-Hop',    'pon el corazón en el trabajo grind todos los días sin parar'),
        ('Country',    'caminos que llevan de vuelta adonde mi corazón siempre pertenecerá'),
        ('R&B',        'tu corazón es todo lo que necesito esta noche quédate aquí conmigo'),
    ],
    'noche': [
        ('Jazz',       'sintiéndome bajo esta noche la trompeta llora por todos mis amores perdidos'),
        ('Electronic', 'la noche entera bailando bajo luces de neón al ritmo del bajo'),
        ('Folk',       'noche tranquila junto al río los sauces susurran tu nombre suave'),
        ('Metal',      'la oscuridad desciende en la noche consumiendo todo lo que vive abajo'),
        ('Pop',        'bailar contigo en la noche hasta que salga el sol mañana'),
    ],
}

print('Calculando embeddings contextuales...')
polisemia_embs = {}
for palabra, contextos in POLISEMIA.items():
    polisemia_embs[palabra] = []
    for genero, oracion in contextos:
        emb = word_contextual_embedding(oracion, palabra)
        polisemia_embs[palabra].append({'genero': genero, 'oracion': oracion, 'emb': emb})
    print(f'  ✓ "{palabra}" — {len(contextos)} contextos')


In [ ]:
fig, axes = plt.subplots(1, len(POLISEMIA), figsize=(18, 5))

for ax, (palabra, resultados) in zip(axes, polisemia_embs.items()):
    n = len(resultados)
    sim = np.zeros((n, n))
    etiquetas = [r['genero'] for r in resultados]
    for i in range(n):
        for j in range(n):
            sim[i, j] = 1 - cosine(resultados[i]['emb'], resultados[j]['emb'])
    im = ax.imshow(sim, cmap='RdYlGn', vmin=0.4, vmax=1.0)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(etiquetas, rotation=35, ha='right', fontsize=8)
    ax.set_yticklabels(etiquetas, fontsize=8)
    ax.set_title(f'"{palabra}"', fontweight='bold', fontsize=11)
    for i in range(n):
        for j in range(n):
            color = 'white' if sim[i,j] < 0.7 else 'black'
            ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center',
                    fontsize=7.5, color=color, fontweight='bold')

plt.colorbar(im, ax=axes[-1], label='Similitud coseno', shrink=0.8)
plt.suptitle('Polisemia contextual con BETO\n'
             'La misma palabra tiene representaciones distintas por género',
             fontsize=12, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(FIGS / 'polisemia_beto.png', dpi=130, bbox_inches='tight')
plt.show()


## BETO Fill-Mask — predicciones por género

In [ ]:
print('Cargando pipeline fill-mask...')
mlm = pipeline('fill-mask', model=MODEL_NAME,
               device=0 if torch.cuda.is_available() else -1)
print('✓ Pipeline listo\n')

MLM_FRASES = {
    'Rock':       'Romperemos las [MASK] y nos alzaremos sobre el dolor esta noche fuerte',
    'Pop':        'Tu amor hace que mi [MASK] lata más rápido cada vez que nos vemos',
    'Hip-Hop':    'Empezamos desde el [MASK] y ahora todo el equipo está arriba trabajando',
    'Metal':      'La [MASK] desciende sobre la tierra consumiendo toda la luz arriba',
    'Electronic': 'Suelta el [MASK] y deja que toda la multitud se mueva al ritmo ahora',
    'Reggaeton':  'Baby mueve tu [MASK] al ritmo del perreo esta noche conmigo aquí',
    'Folk':       'Junto al [MASK] cantamos canciones que nuestros abuelos conocían antes',
}
MLM_FRASES = {g: f for g, f in MLM_FRASES.items() if g in generos_validos}

mlm_resultados = {}
print('=== Predicciones BETO MLM por Género ===')
for genero, frase in MLM_FRASES.items():
    preds = mlm(frase, top_k=5)
    palabras = [p['token_str'].strip() for p in preds]
    probs    = [p['score'] for p in preds]
    mlm_resultados[genero] = {'frase': frase, 'palabras': palabras, 'probs': probs}
    print(f'  {genero:<12} → {palabras}')


In [ ]:
n_generos = len(mlm_resultados)
cols = 2; rows = (n_generos + 1) // 2
fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3.5))
axes = axes.flatten()
colores_genero = dict(zip(mlm_resultados.keys(), PALETTE))

for ax, (genero, datos) in zip(axes, mlm_resultados.items()):
    palabras = datos['palabras'][:5]
    probs    = datos['probs'][:5]
    color    = colores_genero.get(genero, '#999')
    bars = ax.barh(palabras[::-1], probs[::-1], color=color, edgecolor='white', height=0.6)
    ax.set_xlim(0, max(probs) * 1.35)
    frase_corta = datos['frase'].replace('[MASK]', '___')[:65] + '...'
    ax.set_title(f'{genero}\n"{frase_corta}"', fontsize=9, fontweight='bold')
    ax.set_xlabel('Probabilidad', fontsize=9)
    for bar, prob in zip(bars, probs[::-1]):
        ax.text(bar.get_width() + max(probs)*0.02,
                bar.get_y() + bar.get_height()/2,
                f'{prob:.3f}', va='center', fontsize=9)

for ax in axes[len(mlm_resultados):]:
    ax.set_visible(False)

plt.suptitle('BETO Masked LM — Top 5 predicciones por Género',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'mlm_beto_generos.png', dpi=130, bbox_inches='tight')
plt.show()


## Búsqueda semántica de canciones

In [ ]:
def busqueda_semantica(consulta: str, top_k: int = 5) -> pd.DataFrame:
    q_emb = beto_embedding(consulta)
    sims  = np.array([1 - cosine(q_emb, e) for e in beto_embs])
    top_idx = np.argsort(sims)[::-1][:top_k]
    res = df_sample.iloc[top_idx].copy()
    res['Similitud'] = sims[top_idx].round(4)
    return res[['Song', 'Artist', 'Genre', 'Similitud']]


CONSULTAS = [
    ('energía rebelde guitarra distorsionada himno',  '→ Esperado: Rock/Metal'),
    ('amor romántico bailar juntos noche',            '→ Esperado: Pop/R&B'),
    ('calle dinero poder respeto esfuerzo',           '→ Esperado: Hip-Hop'),
    ('oscuridad destrucción caos guerra mal',         '→ Esperado: Metal'),
    ('introspección melancolía acústico tranquilo',   '→ Esperado: Indie/Folk'),
]

print('=== Búsqueda Semántica con BETO ===\n')
for consulta, esperado in CONSULTAS:
    print(f'🔍 "{consulta}"  {esperado}')
    resultados = busqueda_semantica(consulta, top_k=3)
    for _, r in resultados.iterrows():
        print(f'   [{r["Genre"]:10s}] {r["Similitud"]:.4f}  {r["Song"][:35]:35s} – {r["Artist"]}')
    print()
